In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Cell 1 — imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Cell 2 — load data (Kaggle path)
train = pd.read_csv('../input/titanic/train.csv')
test  = pd.read_csv('../input/titanic/test.csv')
print("Train shape:", train.shape, "Test shape:", test.shape)
train.head()


In [ ]:
# Cell 3 — quick EDA counts (optional)
print(train.info())
print(train.describe())
print("Missing in train:\n", train.isnull().sum())
print("Missing in test:\n", test.isnull().sum())


In [ ]:
# Cell 4 — feature engineering function
def extract_title(name):
    # simple title extraction
    import re
    m = re.search(r',\s*([^\.]+)\.', name)
    return m.group(1).strip() if m else "Unknown"

def preprocess(df):
    df = df.copy()
    # Title from Name
    df['Title'] = df['Name'].apply(extract_title)
    # Group rare titles
    rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    df['Title'] = df['Title'].replace(['Mlle','Ms'],'Miss')
    df['Title'] = df['Title'].replace('Mme','Mrs')
    df['Title'] = df['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    # Is alone
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    # Fare log (fill missing with 0 first)
    df['Fare'] = df['Fare'].fillna(0)
    df['Fare_log'] = np.log1p(df['Fare'])
    # Age will be imputed later
    # Simplify cabin: just known/unknown
    df['HasCabin'] = (~df['Cabin'].isnull()).astype(int)
    # Keep a small set of useful columns
    keep_cols = ['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked',
                 'Title','FamilySize','IsAlone','Fare_log','HasCabin']
    return df[keep_cols]

train_p = preprocess(train)
test_p  = preprocess(test)


In [ ]:
# Cell 5 — prepare model pipeline
# numeric and categorical columns
num_cols = ['Age','Fare_log','FamilySize','SibSp','Parch']
cat_cols = ['Pclass','Sex','Embarked','Title','IsAlone','HasCabin']

# Imputer for age (median), and (we already derived Fare_log) — create transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

model = Pipeline(steps=[
    ('preproc', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])


In [ ]:
# Cell 6 — training with cross-validation
X = train_p.copy()
# y = X.pop('Age', None)  # placeholder to ensure no accidental Age-only usage, we'll set X properly

# actual features and target
X = train_p.drop(columns=['PassengerId'])
y = train['Survived']  # use original target

# cross-val
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("CV accuracies:", scores)
print("CV mean accuracy: {:.4f}".format(scores.mean()))


In [ ]:
# Cell 7 — fit on full train and predict test
# model.fit(X, y)

# Prepare test features (ensure same columns)
X_test = test_p.drop(columns=['PassengerId'])
# preds = model.predict(X_test).astype(int)

# make submission DataFrame
submission = pd.DataFrame({
    "PassengerId": test_p['PassengerId'],
    # "Survived": preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created. Shape:", submission.shape)
submission.head()


In [ ]:
# Cell 8 — quick check: ensure exactly 2 columns and right number of rows
# print("Cols:", submission.columns.tolist())
print("Rows (should be 418 for Kaggle Titanic test):", submission.shape[0])
